# 6. Narration

Magi has no NARRATION output: in the dev set it links 72 of the 80 narration lines to some character.
A small logistic regression on simple clues: narration never says "I / me / you", almost never has a tail,
often uses he/she, past tense and names.

Then two steps, because a wrong NARRATION costs a whole extra speaker:
1. does this sequence have a narrator at all? (at least K lines with probability >= G)
2. if yes, every line with probability >= P becomes NARRATION

This notebook also writes the final `predictions.jsonl`. CPU is enough.
Inputs: task data, `magi_output.json` (notebook 1), `char_embeddings.json` (notebook 4), `confirmed.json` (notebook 5).

In [1]:
import os
import re
import sys
import json
import math
import random
import numpy as np
from PIL import Image
from collections import Counter
from functools import lru_cache
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
def find(name):
    for top in ["/kaggle/working", "/kaggle/input", ".."]:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files:
                return root + "/" + name
    return None

DATA = os.path.dirname(find("sequences.json"))
MAGI = find("magi_output.json")
print(DATA)
print(MAGI)
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "../data"

sys.path.insert(0, DATA)
import score
from score import load_jsonl, evaluate, tokens, align_tokens

magi = json.load(open(MAGI))
gold = load_jsonl(DATA + "/development/labels.jsonl", reference=True)
images = {}
for s in json.load(open(DATA + "/sequences.json")):
    images[s["sequence_id"]] = [DATA + "/" + p for p in s["images"]]
dev = sorted(gold.keys())
test = [s for s in magi if magi[s]["split"] == "test"]

emb_file = find("char_embeddings.json")
if emb_file is None:
    emb_file = find("char_embeddings (1).json")     # the name kaggle gave my upload
emb = json.load(open(emb_file))
info = json.load(open(find("confirmed.json")))
PRED_OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else ".."

/kaggle/input/datasets/arpitparashar06/manga-task/manga-task-upload/manga-task/dataset
/kaggle/input/datasets/arpitparashar06/manga-task/magi_output.json


In [3]:
# speed up: the scorer lines up our words with the gold words every time.
# while tuning only the speaker names change, not the text, so remember the line-up.
original_align = score.align_tokens
saved = {}

def remembered_align(g, p):
    key = (tuple(t[0] for t in g), tuple(t[0] for t in p))
    if key not in saved:
        saved[key] = [(a[1], b[1], w) for a, b, w in original_align([(g[i][0], i) for i in range(len(g))],
                                                                    [(p[j][0], j) for j in range(len(p))])]
    return [(g[i], p[j], w) for i, j, w in saved[key]]

score.align_tokens = remembered_align
score.distance = lru_cache(maxsize=None)(score.distance)     # same for the page text comparison

In [4]:
def read_sequence(sid, emb=None):
    # lines = every story line in reading order: [page, text, group]
    # group = (page, magi group) or None if magi linked it to nobody
    lines = []
    groups = {}
    for p in range(3):
        r = magi[sid]["pages"][p]
        link = {a: b for a, b in r["links"]}
        for i in range(len(r["ocr"])):
            if r["keep"][i] == True and r["ocr"][i].strip() != "":
                group = None
                c = link.get(i, -1)
                if 0 <= c < len(r["clusters"]):
                    group = (p, r["clusters"][c])
                    if group not in groups:
                        groups[group] = {"lines": []}
                    groups[group]["lines"].append(len(lines))
                lines.append([p, r["ocr"][i].strip(), group])

        # every group gets one fingerprint: the average of its boxes
        key = sid + "|" + str(p)
        if emb is None or key not in emb or len(emb[key]) == 0:
            continue
        E = np.array(emb[key], dtype=np.float32)
        E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)
        for (gp, g) in groups:
            if gp != p:
                continue
            rows = [c for c in range(min(len(r["clusters"]), len(E))) if r["clusters"][c] == g]
            if len(rows) == 0:
                continue
            v = E[rows].mean(0)
            groups[(p, g)]["vec"] = v / (np.linalg.norm(v) + 1e-8)
    return lines, groups

In [5]:
# notebooks 4 and 5: fingerprints + confirmed speaker rule, T=0.75 CAP=3
T = 0.75
CAP = 3

def confirmed_speaker_rule(lines, inf):
    last = None
    for n in range(len(lines)):
        if not inf[n]["confirmed"] and last is not None:
            lines[n][2] = last
        if inf[n]["confirmed"] and lines[n][2] is not None:
            last = lines[n][2]
    return lines

def merge_groups(groups, T):
    # start with every magi group alone, keep joining the two most alike while they are above T
    keys = sorted([k for k in groups if "vec" in groups[k]])
    clusters = [[k] for k in keys]
    while True:
        best = None
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                s = float(np.mean([groups[a]["vec"] @ groups[b]["vec"] for a in clusters[i] for b in clusters[j]]))
                if s > T and (best is None or s > best[0]):
                    best = (s, i, j)
        if best is None:
            break
        clusters[best[1]] = clusters[best[1]] + clusters[best[2]]
        del clusters[best[2]]
    person = {}
    for n in range(len(clusters)):
        for k in clusters[n]:
            person[k] = "c" + str(n)
    return person

def finish(lines, person, CAP):
    rows = []
    for page, text, g in lines:
        spk = None
        if g is not None:
            spk = person[g] if g in person else "p" + str(g[0]) + "_g" + str(g[1])
        rows.append([page, text, spk])

    # magi linked it to nobody -> previous speaker (or the next one at the start)
    for i in range(len(rows)):
        if rows[i][2] is None:
            spk = None
            for x in range(i - 1, -1, -1):
                if rows[x][2] is not None:
                    spk = rows[x][2]
                    break
            if spk is None:
                for x in range(i + 1, len(rows)):
                    if rows[x][2] is not None:
                        spk = rows[x][2]
                        break
            rows[i][2] = spk if spk is not None else "NARRATION"

    # keep the CAP most talkative, the rest -> previous kept speaker
    count = Counter(r[2] for r in rows if r[2] != "NARRATION")
    keep = [name for name, n in count.most_common(CAP)]
    last = None
    for r in rows:
        if r[2] != "NARRATION" and r[2] not in keep:
            r[2] = last if last is not None else keep[0]
        if r[2] != "NARRATION":
            last = r[2]

    names = {}
    pages = [[], [], []]
    for page, text, spk in rows:
        if spk != "NARRATION":
            if spk not in names:
                names[spk] = "character" + str(len(names) + 1)
            spk = names[spk]
        pages[page].append({"speaker": spk, "text": text})
    return pages

def predict(sid):
    lines, groups = read_sequence(sid, emb)
    lines = confirmed_speaker_rule(lines, info[sid])
    return finish(lines, merge_groups(groups, T), CAP)

## clues for every line

In [6]:
FIRST = re.compile(r"\b(i|i'm|i've|i'll|i'd|me|my|mine|we|we're|us|our|you|you're|your|yours|ya)\b")
THIRD = re.compile(r"\b(he|she|his|her|him|they|their|them|its)\b")
PAST = re.compile(r"\b(was|were|had|did)\b|\w{3,}ed\b")
NAMES = ["tail", "inside", "linked", "first person", "third person", "past tense", "question", "exclamation",
         "names", "digits", "capital share", "words", "box width", "box height", "box x", "box y"]

def clues(sid):
    lines, groups = read_sequence(sid)
    rows = []
    n = 0
    for p in range(3):
        r = magi[sid]["pages"][p]
        W, H = Image.open(images[sid][p]).size
        for i in range(len(r["ocr"])):
            if r["keep"][i] == True and r["ocr"][i].strip() != "":
                text = lines[n][1]
                low = text.lower()
                letters = [c for c in text if c.isalpha()]
                names = re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]{2,}", text)
                b = r["text_boxes"][i]
                rows.append([1.0 if info[sid][n]["tail"] else 0.0,
                             1.0 if info[sid][n]["inside"] else 0.0,
                             1.0 if lines[n][2] is not None else 0.0,
                             1.0 if FIRST.search(low) else 0.0,
                             1.0 if THIRD.search(low) else 0.0,
                             1.0 if PAST.search(low) else 0.0,
                             1.0 if "?" in text else 0.0,
                             1.0 if "!" in text else 0.0,
                             float(min(len(names), 3)),
                             1.0 if re.search(r"\d|:", text) else 0.0,
                             sum(c.isupper() for c in letters) / max(1, len(letters)),
                             float(np.log1p(len(low.split()))),
                             (b[2] - b[0]) / W, (b[3] - b[1]) / H, (b[0] + b[2]) / 2 / W, (b[1] + b[3]) / 2 / H])
                n += 1
    return rows

def answers(sid):
    # gold speaker of a line = the gold speaker most of its words line up with
    lines, groups = read_sequence(sid)
    out = []
    for p in range(3):
        page = [{"speaker": str(n), "text": lines[n][1]} for n in range(len(lines)) if lines[n][0] == p]
        votes = {}
        for g, pr, w in align_tokens(tokens(gold[sid][p]), tokens(page)):
            votes.setdefault(int(pr[1]), Counter())[g[1]] += w
        for x in page:
            k = int(x["speaker"])
            out.append(votes[k].most_common(1)[0][0] if k in votes else None)
    return out

X = {sid: clues(sid) for sid in magi}
y = {sid: answers(sid) for sid in dev}

narr = [row for sid in dev for row, a in zip(X[sid], y[sid]) if a == "NARRATION"]
talk = [row for sid in dev for row, a in zip(X[sid], y[sid]) if a not in (None, "NARRATION")]
print("%-15s narration | character lines" % "")
for f in range(len(NAMES)):
    print("%-15s %.2f      | %.2f" % (NAMES[f], np.mean([r[f] for r in narr]), np.mean([r[f] for r in talk])))

                narration | character lines
tail            0.01      | 0.57
inside          0.85      | 0.95
linked          0.90      | 0.97
first person    0.00      | 0.45
third person    0.31      | 0.12
past tense      0.41      | 0.13
question        0.00      | 0.21
exclamation     0.36      | 0.19
names           0.55      | 0.15
digits          0.04      | 0.02
capital share   0.12      | 0.19
words           2.03      | 1.74
box width       0.12      | 0.10
box height      0.07      | 0.06
box x           0.47      | 0.50
box y           0.52      | 0.50


## the classifier

In [7]:
def train(ids):
    tx, ty = [], []
    for sid in ids:
        for row, a in zip(X[sid], y[sid]):
            if a is not None:                  # UNKNOWN lines have no answer
                tx.append(row)
                ty.append(1 if a == "NARRATION" else 0)
    tx = np.array(tx)
    mean, spread = tx.mean(0), tx.std(0) + 1e-6
    lr = LogisticRegression(max_iter=3000, C=0.5).fit((tx - mean) / spread, ty)
    return {"mean": mean, "spread": spread, "weights": lr.coef_[0], "bias": float(lr.intercept_[0])}

def prob(m, rows):
    if len(rows) == 0:
        return []
    z = ((np.array(rows) - m["mean"]) / m["spread"]) @ m["weights"] + m["bias"]
    return list(1 / (1 + np.exp(-z)))

def add_narration(pages, probs, G, K, P):
    pages = [[dict(x) for x in page] for page in pages]
    if sum(1 for p in probs if p >= G) >= K:          # does this sequence have a narrator?
        n = 0
        for page in pages:
            for x in page:
                if probs[n] >= P:                     # which lines are narration?
                    x["speaker"] = "NARRATION"
                n += 1
    names = {}
    for page in pages:
        for x in page:
            if x["speaker"] != "NARRATION":
                if x["speaker"] not in names:
                    names[x["speaker"]] = "character" + str(len(names) + 1)
                x["speaker"] = names[x["speaker"]]
    return pages

## does it help? honest test

- every dev sequence gets its narration probabilities from a classifier that never saw it (10 folds)
- G, K, P are picked on 4/5 of the sequences and scored on the other 1/5, repeated for 200 shuffles
- the gain moved around depending on how the classifier's 10 folds were cut, so all of it is repeated 6 times

In [8]:
def avg(scores, ids):
    vals = [scores[s] for s in ids if scores[s] is not None]
    return sum(vals) / len(vals)

base = {sid: predict(sid) for sid in dev}

OFF = (9.0, 1, 9.0)                   # narration switched off
settings = [OFF]
for G in [0.4, 0.5, 0.6, 0.7]:
    for K in [1, 2, 3]:
        for P in [0.1, 0.15, 0.2, 0.3, 0.4]:
            if P <= G:
                settings.append((G, K, P))

overall = Counter()                   # score of every setting on all 80, averaged over the 6 repeats
gains = []
for split_seed in range(1, 7):
    ids = list(dev)
    random.Random(split_seed).shuffle(ids)
    oof = {}
    for k in range(10):
        held = ids[k::10]
        m = train([s for s in dev if s not in held])
        for sid in held:
            oof[sid] = prob(m, X[sid])
    pa = [p for sid in dev for p, a in zip(oof[sid], y[sid]) if a is not None]
    ya = [1 if a == "NARRATION" else 0 for sid in dev for a in y[sid] if a is not None]

    res = {}
    for st in settings:
        res[st] = {sid: evaluate({sid: gold[sid]}, {sid: add_narration(base[sid], oof[sid], *st)})["macro"]["balanced_joint_f1"] for sid in dev}
        overall[st] += avg(res[st], dev) / 6

    # gain of every sequence when it is held out, averaged over 200 splits
    gain = Counter()
    for seed in range(200):
        ids = list(dev)
        random.Random(seed).shuffle(ids)
        for k in range(5):
            held = ids[k::5]
            fit = [s for s in ids if s not in held]
            best = max(settings, key=lambda st: avg(res[st], fit))
            for sid in held:
                if res[best][sid] is not None:
                    gain[sid] += (res[best][sid] - res[OFF][sid]) / 200
    g = np.mean([gain[sid] for sid in dev if res[OFF][sid] is not None])
    gains.append(g)
    print("repeat %d: is it narration? AUC %.3f | gain in balanced_joint_f1 %+.3f" % (split_seed, roc_auc_score(ya, pa), g))

print("average gain: %+.3f" % np.mean(gains))

repeat 1: is it narration? AUC 0.929 | gain in balanced_joint_f1 +0.015
repeat 2: is it narration? AUC 0.925 | gain in balanced_joint_f1 +0.007
repeat 3: is it narration? AUC 0.932 | gain in balanced_joint_f1 +0.008
repeat 4: is it narration? AUC 0.933 | gain in balanced_joint_f1 +0.011
repeat 5: is it narration? AUC 0.925 | gain in balanced_joint_f1 +0.013
repeat 6: is it narration? AUC 0.930 | gain in balanced_joint_f1 +0.000
average gain: +0.009


fingerprints + rule was 0.472 (notebook 5), narration adds about +0.01 -> about 0.48.

## final predictions

G, K, P = the setting that works best on average over the 6 repeats. the classifier is trained on all 80 dev sequences.

In [9]:
G, K, P = max(settings, key=lambda st: overall[st])
print("G=%s K=%s P=%s" % (G, K, P))
narration_model = train(dev)

def final(sid):
    return add_narration(predict(sid), prob(narration_model, X[sid]), G, K, P)

f = open(PRED_OUT + "/predictions.jsonl", "w")
for sid in test:
    f.write(json.dumps({"sequence_id": sid, "pages": final(sid)}) + "\n")
f.close()
print("saved predictions.jsonl:", len(test), "test sequences")

scores = evaluate(gold, {sid: final(sid) for sid in dev})["macro"]
print("dev scores (settings picked on dev, so a bit optimistic):")
for k in scores:
    print("  %-30s %.3f" % (k, scores[k]))

G=0.6 K=1 P=0.15
saved predictions.jsonl: 15 test sequences
dev scores (settings picked on dev, so a bit optimistic):
  text_order_score               0.929
  balanced_joint_f1              0.510
  speaker_accuracy_on_matched    0.712
  joint_f1                       0.673
  matched_token_coverage         0.963


In [ ]:
# save the classifier and G, K, P for predict.py
to_save = {"mean": narration_model["mean"].tolist(),
           "spread": narration_model["spread"].tolist(),
           "weights": narration_model["weights"].tolist(),
           "bias": narration_model["bias"],
           "features": NAMES,
           "G": G, "K": K, "P": P}
json.dump(to_save, open(PRED_OUT + "/narration_model.json", "w"), indent=1)
print("saved narration_model.json")